## Process Region and Department Datasets
Dataset source: https://data-interne.ademe.fr/datasets/ref-regions-domcom-rapproches

In [ ]:
import os
from pathlib import Path
from loguru import logger
import geopandas as gpd

# Move to the root directory of the project
os.chdir(Path.cwd().parent)
logger.info("Current working directory : {}", Path.cwd())

### Régions 

In [ ]:
region = gpd.read_file("data/geojson/input/ref-regions-domcom-rapproches.geojson")
region

In [ ]:
region = region.rename(
    columns={
        "DREG_L_LIB": "nom_reg",
        "DREG_C_COD": "code_reg",
        "geometry": "geometry_region",
    }
).drop(columns=["_geopoint", "id", "_id", "_i"])
region

In [ ]:
print(type(region))
region.to_file("data/geojson/output/region.geojson", driver="GeoJSON")

### Départements

In [ ]:
departement = gpd.read_file(
    "data/geojson/input/ref-departements-domcom-rapproches.geojson"
)
departement

In [ ]:
departement = departement.rename(
    columns={
        "DDEP_C_COD": "code_dep",
        "DREG_L_LIB": "nom_reg",
        "DDEP_L_LIB": "nom_dep",
        "geometry": "geometry_departement",
    }
).drop(columns=["id", "_geopoint", "_id"])
departement["code_dep"] = departement["code_dep"].replace({"2A": "20", "2B": "20"})

In [ ]:
print(type(departement))
departement.to_file("data/geojson/output/departement.geojson", driver="GeoJSON")

### Merge Departement - Région

In [ ]:
reg_dep = departement.merge(
    region, how="left", left_on="nom_reg", right_on="nom_reg"
).sort_values("code_reg")
reg_dep

In [ ]:
reg_dep[reg_dep["code_dep"] == "20"]

In [ ]:
reg_dep.groupby(["nom_reg", "code_reg"]).size().reset_index()

In [ ]:
reg_dep = gpd.GeoDataFrame(reg_dep, geometry="geometry_departement")

In [ ]:
print(type(reg_dep))
reg_dep.to_parquet("data/geojson/output/region_departement.parquet")